In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from harbor.analysis.cross_docking import DockingDataModel
from plotting_params import *

## input files

In [ ]:
posit_raw_df = pd.read_parquet("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/ALL_combined_results.parquet")

In [ ]:
posit_results = Path("x_to_y_posit_combined_results.csv")
analyzed_dir = Path("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/analyzed_results")
pdf = pd.concat([pd.read_csv(analyzed_dir / csv) for csv in ["x_to_y_posit_combined_results.csv", "x_to_x_posit_combined_results.csv", "x_to_y_posit_5_refs_combined_results.csv", "x_to_x_posit_5_refs_combined_results.csv"]])

In [ ]:
pdf.nunique()

In [ ]:
pdf["Error_Lower"] = pdf["Fraction"] - pdf["CI_Lower"]
pdf["Error_Lower"] = pdf["Error_Lower"].apply(lambda x: 0 if x < 0 else x)
pdf["Error_Upper"] = pdf["CI_Upper"] - pdf["Fraction"]
pdf["Error_Upper"] = pdf["Error_Upper"].apply(lambda x: 0 if x < 0 else x)

## output files

In [ ]:
figpath = Path("../figures")
figpath.mkdir(exist_ok=True)

# Scaffold Split

In [ ]:
sdf = pdf[(pdf["PairwiseSplit"] == "ScaffoldSplit")&(pdf["Scaffold_Split_Option"].isin(['x_to_y', 'x_to_x']))]

In [ ]:
# replace brackets in the query and ref columns
query = "Query_Scaffold_ID_Subset"
ref = "Reference_Scaffold_ID_Subset"
sdf[query] = (
    sdf[query]
    .astype(str)
    .apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
)
sdf["qint"] = sdf[query].astype(float).tolist()
sdf[ref] = (
    sdf[ref]
    .astype(str)
    .apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
)
sdf["rint"] = sdf[ref].astype(float).tolist()
# sdf = sdf.groupby(["Score", "qint", 'rint']).head(1)

In [ ]:
sdf

In [ ]:
sdf[(sdf["rint"] == 1)&(sdf["qint"] == 1)]

## Scaffold x_to_y heatmap - all refs

In [ ]:
ssdf = sdf[sdf["Reference_Split"].isna()]

In [ ]:
heatmap_dfs = {
        "_".join(name): group for name, group in ssdf.groupby(["Score"])
    }

In [ ]:
def get_label(var):
    return label_map.get(var,var)

In [ ]:
fig_name = "scaffold_x_to_y_heatmap"
for name, heatmap_df in heatmap_dfs.items():
    pivot_fraction = heatmap_df.pivot(
        index="qint", columns="rint", values="Fraction"
    )
    ref_counts = (
        heatmap_df.sort_values("rint")
        .groupby(ref)
        .head(1)[[ref, "Total"]]
        .to_dict(orient="records")
    )
    count_dict = {data[ref]: data["Total"] for data in ref_counts}

    query_counts = (
        heatmap_df.sort_values("qint")
        .groupby(query)
        .head(1)[[query, "Total"]]
        .to_dict(orient="records")
    )
    count_dict = {data[query]: data["Total"] for data in query_counts}
    
    ytick_labels = [
        f"$\\bf{int(cluster_id) + 1}$ ({total})" for cluster_id, total in count_dict.items()
    ]
    xtick_labels = [
        f"$\\bf{int(cluster_id) + 1}$\n({total})" for cluster_id, total in count_dict.items()
    ]
    plt.figure(figsize=LARGE_FIG_SIZE)
    
    # Create custom annotation array
    annotations = pivot_fraction.copy()
    annotations = annotations.map(lambda x: '' if x in [0.0] else f'{x:.1f}')
    heatmap = sns.heatmap(
        data=pivot_fraction,
        xticklabels=xtick_labels,
        yticklabels=ytick_labels,
        annot=annotations,
        fmt='',
        cmap="coolwarm_r",
        vmin=0, 
        vmax=1
    )
    # Add colorbar label
    heatmap.collections[0].colorbar.set_label("Fraction")

    # Rotate axis labels for better readability
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)

    # Invert y-axis to put 0 at bottom
    plt.gca().invert_yaxis()

    # Set axis labels
    plt.xlabel(
        f"$\\bf{{Reference Scaffold ID}}$\n (# Reference Structures with Scaffold)",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="normal",
    )
    plt.ylabel(
        f"$\\bf{{Query Scaffold ID}}$\n (# Query Ligands with Scaffold)",
        fontsize=FONT_SIZES["ylabel"],
        fontweight="normal",
    )
    plt.title(
        f"Scored by {get_label(name)}",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="bold",
    )

    plt.savefig(figpath / f"{fig_name}_{name}.svg", format="svg", bbox_inches="tight")
    plt.savefig(figpath / f"{fig_name}_{name}.png", format="png", bbox_inches="tight")

### sort by size of scaffold

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

def draw_scaffold(mol, output_path, size=(400, 400), bond_length=30):
    # Set drawing options
    opts = Draw.DrawingOptions()
    opts.bondLength = bond_length  # Fixed bond length for all molecules
    opts.fixedBondLength = bond_length
    opts.coordScale = 1.0
    
    # Draw the molecule with consistent bond length
    img = Draw.MolToImage(
        mol,
        size=size,
        options=opts,
        # kekulize=True,
        # fitImage=True,    # Auto-scales to fit the image size while maintaining bond length ratio
        imageType="png"
    )
    
    img.save(output_path)

def get_scaffold_size(mol):    
    return mol.GetNumHeavyAtoms()

In [ ]:
size_dict = {}
mols = []
sizes = []
names = []
counts = []
legends = []
for i in range(19):
    scaffold_smiles = posit_raw_df[posit_raw_df.cluster_id == i].groupby(["Query_Ligand"]).head(1).scaffold_smarts.unique()[0]
    mol = Chem.MolFromSmiles(scaffold_smiles)
    names.append(f"Scaffold_{i}")
    size_dict[i] = mol.GetNumHeavyAtoms()
    sizes.append(mol.GetNumHeavyAtoms())
    counts.append(count_dict[str(i)])
    mols.append(mol)
    legends.append(f"Scaffold_{i+1} # Molecules: {count_dict[str(i)]} # Atoms: {mol.GetNumHeavyAtoms()}")
    draw_scaffold(mol, figpath / f"scaffold_{i}.png")

In [ ]:
moldf = pd.DataFrame({"Mol": mols, "Name": names, "Counts": counts, "Size": sizes, "Legend":legends})

In [ ]:
opts = Draw.MolDrawOptions()
opts.legendFraction = 0.25
opts.legendFontSize = 18
img = Draw.MolsToGridImage(moldf.Mol.tolist(), 
                           molsPerRow=4, 
                           legends=moldf.Legend.tolist(), 
                           subImgSize=(400, 200),
                           drawOptions=opts,
                           useSVG=True)
with open(figpath / "scaffold_grid.svg", 'w') as f:
    f.write(img.data)

In [ ]:
img = Draw.MolsToGridImage(moldf.Mol.tolist(), 
                           molsPerRow=4,  
                           subImgSize=(400, 200),
                           drawOptions=opts,
                           useSVG=True)
with open(figpath / "scaffold_grid_no_legend.svg", 'w') as f:
    f.write(img.data)

In [ ]:
sorted_clusters = sorted(size_dict.items(), key=lambda x: x[1], reverse=True)
cluster_order = [x[0] for x in sorted_clusters]

In [ ]:
fig_name = "scaffold_x_to_y_sorted_by_size"
for name, heatmap_df in heatmap_dfs.items():
    # Sort clusters by size
    pivot_fraction = heatmap_df.pivot(
        index="qint", columns="rint", values="Fraction"
    )
    
    # Reorder columns and index according to size
    pivot_fraction = pivot_fraction.reindex(columns=cluster_order)
    pivot_fraction = pivot_fraction.reindex(cluster_order)
    
    ref_counts = (
        heatmap_df.sort_values("rint")
        .groupby(ref)
        .head(1)[[ref, "Total"]]
        .to_dict(orient="records")
    )
    count_dict = {int(data[ref]): data["Total"] for data in ref_counts}

    # Create labels in size order
    ytick_labels = [
        f"$\\bf{int(cluster_id) + 1}$ ({size_dict[cluster_id]})" 
        for cluster_id in cluster_order
    ]
    xtick_labels = [
        f"$\\bf{int(cluster_id) + 1}$\n({size_dict[cluster_id]})" 
        for cluster_id in cluster_order
    ]
    
    # ytick_labels = [
    #     f"$\\bf{int(cluster_id) + 1}$" 
    #     for cluster_id in cluster_order
    # ]
    # xtick_labels = [
    #     f"$\\bf{int(cluster_id) + 1}$" 
    #     for cluster_id in cluster_order
    # ]
    
    plt.figure(figsize=LARGE_FIG_SIZE)

    # Create custom annotation array
    annotations = pivot_fraction.copy()
    annotations = annotations.map(lambda x: '' if x in [0.0] else f'{x:.1f}')
    
    heatmap = sns.heatmap(
        data=pivot_fraction,
        xticklabels=xtick_labels,
        yticklabels=ytick_labels,
        annot=annotations,
        fmt='',
        cmap="coolwarm_r",
        vmin=0, 
        vmax=1
    )
    # Add colorbar label
    heatmap.collections[0].colorbar.set_label(Y_LABEL, fontsize=FONT_SIZES["legend_title"], fontweight='bold')

    # Rotate axis labels for better readability
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)

    # Invert y-axis to put 0 at bottom
    plt.gca().invert_yaxis()

    # Set axis labels
    plt.xlabel(
        f"$\\bf{{Reference}}$ $\\bf{{Scaffold}}$ $\\bf{{ID}}$ \n (# Heavy Atoms in Scaffold)",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="normal",
    )
    plt.ylabel(
        f"$\\bf{{Query}}$ $\\bf{{Scaffold}}$ $\\bf{{ID}}$ \n (# Heavy Atoms in Scaffold)",
        fontsize=FONT_SIZES["ylabel"],
        fontweight="normal",
    )
    # plt.xlabel(
    #     f"Reference Scaffold ID",
    #     fontsize=FONT_SIZES["xlabel"],
    #     fontweight="bold",
    # )
    # plt.ylabel(
    #     f"Query Scaffold ID",
    #     fontsize=FONT_SIZES["ylabel"],
    #     fontweight="bold",
    # )
    plt.title(
        f"Scored by {get_label(name)}",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="bold",
    )

    plt.savefig(figpath / f"{fig_name}_{name}.svg", format="svg", bbox_inches="tight")
    plt.savefig(figpath / f"{fig_name}_{name}.png", format="png", bbox_inches="tight")

## Scaffold x_to_y heatmap - 5 refs

In [ ]:
ssdf = sdf[sdf["N_Reference_Structures"] == 5]

In [ ]:
heatmap_dfs = {
        "_".join(name): group for name, group in ssdf.groupby(["Score"])
    }

In [ ]:
def get_label(var):
    return label_map.get(var,var)

In [ ]:
fig_name = "scaffold_x_to_y_heatmap_5_refs"
for name, heatmap_df in heatmap_dfs.items():
    pivot_fraction = heatmap_df.pivot(
        index="qint", columns="rint", values="Fraction"
    )
    ref_counts = (
        heatmap_df.sort_values("rint")
        .groupby(ref)
        .head(1)[[ref, "Total"]]
        .to_dict(orient="records")
    )
    count_dict = {data[ref]: data["Total"] for data in ref_counts}

    query_counts = (
        heatmap_df.sort_values("qint")
        .groupby(query)
        .head(1)[[query, "Total"]]
        .to_dict(orient="records")
    )
    count_dict = {data[query]: data["Total"] for data in query_counts}
    
    ytick_labels = [
        f"$\\bf{int(cluster_id) + 1}$ ({total})" for cluster_id, total in count_dict.items()
    ]
    xtick_labels = [
        f"$\\bf{int(cluster_id) + 1}$\n({total})" for cluster_id, total in count_dict.items()
    ]
    plt.figure(figsize=LARGE_FIG_SIZE)
    
    # Create custom annotation array
    annotations = pivot_fraction.copy()
    annotations = annotations.map(lambda x: '' if x in [0.0] else f'{x:.1f}')
    heatmap = sns.heatmap(
        data=pivot_fraction,
        xticklabels=xtick_labels,
        yticklabels=ytick_labels,
        annot=annotations,
        fmt='',
        cmap="coolwarm_r",
        vmin=0, 
        vmax=1
    )
    # Add colorbar label
    heatmap.collections[0].colorbar.set_label("Fraction")

    # Rotate axis labels for better readability
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)

    # Invert y-axis to put 0 at bottom
    plt.gca().invert_yaxis()

    # Set axis labels
    plt.xlabel(
        f"$\\bf{{Reference Scaffold ID}}$\n (# Reference Structures with Scaffold)",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="normal",
    )
    plt.ylabel(
        f"$\\bf{{Query Scaffold ID}}$\n (# Query Ligands with Scaffold)",
        fontsize=FONT_SIZES["ylabel"],
        fontweight="normal",
    )
    plt.title(
        f"Scored by {get_label(name)}",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="bold",
    )

    plt.savefig(figpath / f"{fig_name}_{name}.svg", format="svg", bbox_inches="tight")
    plt.savefig(figpath / f"{fig_name}_{name}.png", format="png", bbox_inches="tight")

### sort by size of scaffold

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

def draw_scaffold(mol, output_path, size=(400, 400), bond_length=30):
    # Set drawing options
    opts = Draw.DrawingOptions()
    opts.bondLength = bond_length  # Fixed bond length for all molecules
    opts.fixedBondLength = bond_length
    opts.coordScale = 1.0
    
    # Draw the molecule with consistent bond length
    img = Draw.MolToImage(
        mol,
        size=size,
        options=opts,
        # kekulize=True,
        # fitImage=True,    # Auto-scales to fit the image size while maintaining bond length ratio
        imageType="png"
    )
    
    img.save(output_path)

def get_scaffold_size(mol):    
    return mol.GetNumHeavyAtoms()

In [ ]:
size_dict = {}
mols = []
sizes = []
names = []
counts = []
legends = []
for i in range(19):
    scaffold_smiles = posit_raw_df[posit_raw_df.cluster_id == i].groupby(["Query_Ligand"]).head(1).scaffold_smarts.unique()[0]
    mol = Chem.MolFromSmiles(scaffold_smiles)
    names.append(f"Scaffold_{i}")
    size_dict[i] = mol.GetNumHeavyAtoms()
    sizes.append(mol.GetNumHeavyAtoms())
    counts.append(count_dict[str(i)])
    mols.append(mol)
    legends.append(f"Scaffold_{i+1} # Molecules: {count_dict[str(i)]} # Atoms: {mol.GetNumHeavyAtoms()}")
    draw_scaffold(mol, figpath / f"scaffold_{i}.png")

In [ ]:
moldf = pd.DataFrame({"Mol": mols, "Name": names, "Counts": counts, "Size": sizes, "Legend":legends})

In [ ]:
opts = Draw.MolDrawOptions()
opts.legendFraction = 0.25
opts.legendFontSize = 18
img = Draw.MolsToGridImage(moldf.Mol.tolist(), 
                           molsPerRow=4, 
                           legends=moldf.Legend.tolist(), 
                           subImgSize=(400, 200),
                           drawOptions=opts,
                           useSVG=True)
with open(figpath / "scaffold_grid.svg", 'w') as f:
    f.write(img.data)

In [ ]:
img = Draw.MolsToGridImage(moldf.Mol.tolist(), 
                           molsPerRow=4,  
                           subImgSize=(400, 200),
                           drawOptions=opts,
                           useSVG=True)
with open(figpath / "scaffold_grid_no_legend.svg", 'w') as f:
    f.write(img.data)

In [ ]:
sorted_clusters = sorted(size_dict.items(), key=lambda x: x[1], reverse=True)
cluster_order = [x[0] for x in sorted_clusters]

In [ ]:
fig_name = "scaffold_x_to_y_sorted_by_size_5_refs"
for name, heatmap_df in heatmap_dfs.items():
    # Sort clusters by size
    pivot_fraction = heatmap_df.pivot(
        index="qint", columns="rint", values="Fraction"
    )
    
    # Reorder columns and index according to size
    pivot_fraction = pivot_fraction.reindex(columns=cluster_order)
    pivot_fraction = pivot_fraction.reindex(cluster_order)
    
    ref_counts = (
        heatmap_df.sort_values("rint")
        .groupby(ref)
        .head(1)[[ref, "Total"]]
        .to_dict(orient="records")
    )
    count_dict = {int(data[ref]): data["Total"] for data in ref_counts}

    # Create labels in size order
    ytick_labels = [
        f"$\\bf{int(cluster_id) + 1}$ ({size_dict[cluster_id]})" 
        for cluster_id in cluster_order
    ]
    xtick_labels = [
        f"$\\bf{int(cluster_id) + 1}$\n({size_dict[cluster_id]})" 
        for cluster_id in cluster_order
    ]
    
    # ytick_labels = [
    #     f"$\\bf{int(cluster_id) + 1}$" 
    #     for cluster_id in cluster_order
    # ]
    # xtick_labels = [
    #     f"$\\bf{int(cluster_id) + 1}$" 
    #     for cluster_id in cluster_order
    # ]
    
    plt.figure(figsize=LARGE_FIG_SIZE)

    # Create custom annotation array
    annotations = pivot_fraction.copy()
    annotations = annotations.map(lambda x: '' if x in [0.0] else f'{x:.1f}')
    
    heatmap = sns.heatmap(
        data=pivot_fraction,
        xticklabels=xtick_labels,
        yticklabels=ytick_labels,
        annot=annotations,
        fmt='',
        cmap="coolwarm_r",
        vmin=0, 
        vmax=1
    )
    # Add colorbar label
    heatmap.collections[0].colorbar.set_label(Y_LABEL, fontsize=FONT_SIZES["legend_title"], fontweight='bold')

    # Rotate axis labels for better readability
    plt.xticks(rotation=0)
    plt.yticks(rotation=0)

    # Invert y-axis to put 0 at bottom
    plt.gca().invert_yaxis()

    # Set axis labels
    plt.xlabel(
        f"$\\bf{{Reference}}$ $\\bf{{Scaffold}}$ $\\bf{{ID}}$ \n (# Heavy Atoms in Scaffold)",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="normal",
    )
    plt.ylabel(
        f"$\\bf{{Query}}$ $\\bf{{Scaffold}}$ $\\bf{{ID}}$ \n (# Heavy Atoms in Scaffold)",
        fontsize=FONT_SIZES["ylabel"],
        fontweight="normal",
    )
    # plt.xlabel(
    #     f"Reference Scaffold ID",
    #     fontsize=FONT_SIZES["xlabel"],
    #     fontweight="bold",
    # )
    # plt.ylabel(
    #     f"Query Scaffold ID",
    #     fontsize=FONT_SIZES["ylabel"],
    #     fontweight="bold",
    # )
    plt.title(
        f"Scored by {get_label(name)}",
        fontsize=FONT_SIZES["xlabel"],
        fontweight="bold",
    )

    plt.savefig(figpath / f"{fig_name}_{name}.svg", format="svg", bbox_inches="tight")
    plt.savefig(figpath / f"{fig_name}_{name}.png", format="png", bbox_inches="tight")